In [1]:
import torch
import numpy as np
from sklearn.metrics import f1_score
from transformers import pipeline
from datasets import load_dataset
from huggingface_hub import login
from datasets.arrow_dataset import Dataset
from datasets.dataset_dict import DatasetDict, IterableDatasetDict
from datasets.iterable_dataset import IterableDataset
from transformers import AutoTokenizer
from transformers import AutoModelForSequenceClassification
from transformers import Trainer, TrainingArguments

c:\WORK\textClassifier\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import transformers, torch, accelerate, huggingface_hub
print(transformers.__version__)
print(torch.__version__)
print(accelerate.__version__)
print(huggingface_hub.__version__)

5.3.0
2.10.0+cpu
1.13.0
1.6.0


In [ ]:

login()

In [4]:
# Dataset id from huggingface.co/dataset
dataset_id = "argilla/synthetic-domain-text-classification"
 
# Load raw dataset
train_dataset = load_dataset(dataset_id, split='train')

split_dataset = train_dataset.train_test_split(test_size=0.1)
split_dataset['train'][0]
# {'text': 'Recently, there has been an increase in property values within the suburban areas of several cities due to improvements in infrastructure and lifestyle amenities such as parks, retail stores, and educational institutions nearby. Additionally, new housing developments are emerging, catering to different family needs with varying sizes and price ranges. These changes have influenced investment decisions for many looking to buy or sell properties.', 'label': 14}

{'text': 'An exploration of how educational systems in different countries promote lifelong learning opportunities for their citizens. This article discusses the importance of continuous skill development and training programs in adapting to changes in job markets and technological advancements. It also examines various approaches employed by governments, institutions, and employers in fostering a culture of education and personal growth.',
 'label': 16}

In [5]:
# Model id to load the tokenizer
model_id = "answerdotai/ModernBERT-base"

# Load Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id)
 
# Tokenize helper function
def tokenize(batch):
    return tokenizer(batch['text'], padding=True, truncation=True, return_tensors="pt")
 
# Tokenize dataset
if "label" in split_dataset["train"].features.keys():
    split_dataset =  split_dataset.rename_column("label", "labels") # to match Trainer
tokenized_dataset = split_dataset.map(tokenize, batched=True, remove_columns=["text"])
 
tokenized_dataset["train"].features.keys()
# dict_keys(['labels', 'input_ids', 'attention_mask'])

Map: 100%|██████████| 100/100 [00:00<00:00, 1863.66 examples/s]


dict_keys(['labels', 'input_ids', 'attention_mask'])

In [6]:
# Prepare model labels - useful for inference
labels = tokenized_dataset["train"].features["labels"].names
num_labels = len(labels)
label2id, id2label = dict(), dict()
for i, label in enumerate(labels):
    label2id[label] = str(i)
    id2label[str(i)] = label
 
# Download the model from huggingface.co/models
model = AutoModelForSequenceClassification.from_pretrained(
    model_id, num_labels=num_labels, label2id=label2id, id2label=id2label,
)

Loading weights: 100%|██████████| 136/136 [00:00<00:00, 2420.51it/s]
ModernBertForSequenceClassification LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.weight | MISSING    | 
classifier.bias   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [7]:
# Metric helper method
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    score = f1_score(
            labels, predictions, labels=labels, pos_label=1, average="weighted"
        )
    return {"f1": float(score) if score == 1 else score}

In [8]:
# Define training args
training_args = TrainingArguments(
    output_dir="ModernBERT-domain-classifier",
    # smaller batches for CPU
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    learning_rate=5e-5,
    num_train_epochs=5,
    # disable GPU precision tricks
    bf16=False,
    fp16=False,
    # CPU compatible optimizer
    optim="adamw_torch",
    # logging & evaluation
    logging_strategy="steps",
    logging_steps=100,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    # hub
    push_to_hub=True,
    hub_strategy="every_save",
    hub_token=None,
)
 
# Create a Trainer instance
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    compute_metrics=compute_metrics,
)
trainer.train()

c:\WORK\EncoderTextClassifier\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,F1
1,0.733935,0.679565,0.894462
2,0.338235,0.272681,0.921166
3,0.051224,0.529678,0.915418
4,0.053466,0.520237,0.912864
5,0.000978,0.565501,0.912864


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.21it/s]
c:\WORK\EncoderTextClassifier\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.49s/it]
c:\WORK\EncoderTextClassifier\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.55s/it]
c:\WORK\EncoderTextClassifier\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.39s/it]
c:\WORK\EncoderTe

TrainOutput(global_step=1125, training_loss=0.35426912649431164, metrics={'train_runtime': 14272.5596, 'train_samples_per_second': 0.315, 'train_steps_per_second': 0.079, 'total_flos': 1120294678932000.0, 'train_loss': 0.35426912649431164, 'epoch': 5.0})

In [ ]:
# load model from huggingface.co/models using our repository id

# tokenizer = AutoTokenizer.from_pretrained("answerdotai/ModernBERT-base")

classifier = pipeline(
    task="text-classification", 
    model="ModernBERT-domain-classifier", 
    device=0,
    tokenizer=tokenizer
)
 
sample = "Filler produced a stunning escape of his own to narrow the gap before Woodward’s missed cut allowed the Germans to level. A missed 9-ball from Woodward and a scratch from Gorst handed Europe control, and despite escaping a safety, Gorst’s miss on the 3-ball sealed it for the Europeans."
 
classifier(sample)

KeyError: "Unknown task text-classificatibon, available tasks are ['any-to-any', 'audio-classification', 'automatic-speech-recognition', 'depth-estimation', 'document-question-answering', 'feature-extraction', 'fill-mask', 'image-classification', 'image-feature-extraction', 'image-segmentation', 'image-text-to-text', 'keypoint-matching', 'mask-generation', 'ner', 'object-detection', 'sentiment-analysis', 'table-question-answering', 'text-classification', 'text-generation', 'text-to-audio', 'text-to-speech', 'token-classification', 'video-classification', 'zero-shot-audio-classification', 'zero-shot-classification', 'zero-shot-image-classification', 'zero-shot-object-detection']"

In [15]:
len(train_dataset)

1000